<a href="https://colab.research.google.com/github/Harshada900/pyspark-practise/blob/main/8_UDF_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName('Basics').getOrCreate()

### **Step 1 - write normal python function**

In [2]:
def categorize_salary(salary):
  if salary is None:
    return None
  elif salary >=80000:
    return 'High'
  elif salary >= 60000:
    return 'Medium'
  else:
    return 'Low'

### **Step 2 - register it as udf**

In [3]:
from pyspark.sql.functions import udf, col
from pyspark.sql.types import StringType

# type hint

salary_udf = udf(categorize_salary, StringType())


### **Step 3 - use it like any other column function**

In [4]:
df = spark.createDataFrame([
    (1, "Raj",   50000.0),
    (2, "Priya", 70000.0),
    (3, "Sara",  90000.0),
    (4, "Amit",  None)
], ["emp_id", "name", "salary"])

In [5]:
df.withColumn("salary_category", salary_udf(col("salary"))).show()

+------+-----+-------+---------------+
|emp_id| name| salary|salary_category|
+------+-----+-------+---------------+
|     1|  Raj|50000.0|            Low|
|     2|Priya|70000.0|         Medium|
|     3| Sara|90000.0|           High|
|     4| Amit|   NULL|           NULL|
+------+-----+-------+---------------+



# **Decorator style - cleaner way**

In [6]:
# Define and register in one step using decorator
@udf(StringType())
def categorize_salary(salary):
  if salary is None:
    return None
  elif salary >=80000:
    return 'High'
  elif salary >= 60000:
    return 'Medium'
  else:
    return 'Low'

# Use it directly
df.withColumn("salary_category", categorize_salary(col("salary"))).show()

+------+-----+-------+---------------+
|emp_id| name| salary|salary_category|
+------+-----+-------+---------------+
|     1|  Raj|50000.0|            Low|
|     2|Priya|70000.0|         Medium|
|     3| Sara|90000.0|           High|
|     4| Amit|   NULL|           NULL|
+------+-----+-------+---------------+



## **Registering UDF for SQL**

In [7]:
# step 1 -Register with a name for SQL use

spark.udf.register("categorize_salary_sql", categorize_salary)

# Step 2 — Create a temp view from your DataFrame
df.createOrReplaceTempView("employees")

#Now use in sql

spark.sql("""
 SELECT name, salary, categorize_salary_sql(salary) as salary_category
 FROM employees
""").show()

+-----+-------+---------------+
| name| salary|salary_category|
+-----+-------+---------------+
|  Raj|50000.0|            Low|
|Priya|70000.0|         Medium|
| Sara|90000.0|           High|
| Amit|   NULL|           NULL|
+-----+-------+---------------+



### **UDF with Multiple Arguments**

In [8]:
df2 = spark.createDataFrame([
    ( "Raj", "P"),
    ( "Priya", "A"),
    ( "Sara",  "K"),
    ( "Amit",  None)
], ["first_name", "last_name"])

@udf(StringType())
def full_name(first, last):
    if first is None or last is None:
        return None
    return f"{first} {last}"

# Pass multiple columns
df2.withColumn("full_name", full_name(col("first_name"), col("last_name"))).show()

+----------+---------+---------+
|first_name|last_name|full_name|
+----------+---------+---------+
|       Raj|        P|    Raj P|
|     Priya|        A|  Priya A|
|      Sara|        K|   Sara K|
|      Amit|     NULL|     NULL|
+----------+---------+---------+



## **The Better Alternative — Pandas UDF (Vectorized UDF)**

In [9]:
from pyspark.sql.functions import pandas_udf
import pandas as pd

In [10]:
# pandas_udf receives entire column as pandas Series
# processes whole batch at once — much faster!

# Create a fresh clear DataFrame
df = spark.createDataFrame([
    (1, "Raj",   50000.0),
    (2, "Priya", 70000.0),
    (3, "Sara",  90000.0),
    (4, "Amit",  None)
], ["emp_id", "name", "salary"])

@pandas_udf(StringType())
def categorize_salary_pandas(salary: pd.Series) -> pd.Series:
  return salary.apply(lambda x:
                      "unknown" if pd.isna(x)
                      else "High" if x>=80000
                      else "Medium" if x>=60000
                      else "Low"
                      )


In [14]:

result = df.withColumn("salary_category",
                        categorize_salary_pandas(col("salary")))
result.show()

+------+-----+-------+---------------+
|emp_id| name| salary|salary_category|
+------+-----+-------+---------------+
|     1|  Raj|50000.0|            Low|
|     2|Priya|70000.0|         Medium|
|     3| Sara|90000.0|           High|
|     4| Amit|   NULL|        unknown|
+------+-----+-------+---------------+



In [15]:
df.withColumn("salary_category", categorize_salary_pandas(col("salary"))).show()

+------+-----+-------+---------------+
|emp_id| name| salary|salary_category|
+------+-----+-------+---------------+
|     1|  Raj|50000.0|            Low|
|     2|Priya|70000.0|         Medium|
|     3| Sara|90000.0|           High|
|     4| Amit|   NULL|        unknown|
+------+-----+-------+---------------+

